# Performance Rating Prediction — Model HR Ketiga

Use case **#4** di `USE_CASES.md`: memprediksi **kategori kinerja** karyawan
(Rendah / Sedang / Tinggi) untuk membantu perencanaan pengembangan, bukan
untuk menghukum.

> **Data:** `data/hr_performance.csv` adalah dataset **sintetis**
> (`scripts/generate_performance_data.py`).

## Catatan penting soal target

`kategori_kinerja` berasal dari `rating_kinerja` yang **ordinal** (1–5).
Karena itu selain akurasi/macro-F1, dihitung juga **MAE ordinal** (jarak rata-rata
prediksi ke nilai sebenarnya) — jauh lebih bermakna daripada akurasi saja.

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    mean_absolute_error, cohen_kappa_score,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["data/hr_performance.csv", "../data/hr_performance.csv", "hr_performance.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_performance.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
ARTIFACT_DIR = PROJECT_ROOT / "models" / "performance" / "v1"

TARGET = "kategori_kinerja"
ORDER = ["Rendah", "Sedang", "Tinggi"]
LABEL2NUM = {k: i + 1 for i, k in enumerate(ORDER)}   # 1..3 ordinal
LEAKAGE = ["employee_id", "rating_kinerja", TARGET]

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 170)
print("Data :", DATA_PATH)
print("Artefak:", ARTIFACT_DIR)

Data : data/hr_performance.csv
Artefak: /workspace/project/hr-intelligence-system/models/performance/v1


## 1. Load & Audit

In [2]:
df = pd.read_csv(DATA_PATH)

FEATURES = [c for c in df.columns if c not in LEAKAGE]
NUM_COLS = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = [c for c in FEATURES if c not in NUM_COLS]

print("Baris:", len(df), "| fitur:", len(FEATURES))
print("Numerik :", NUM_COLS)
print("Kategori:", CAT_COLS)
print("\nDistribusi target:")
print(df[TARGET].value_counts().reindex(ORDER).to_string())
df[FEATURES + [TARGET]].head(3)

Baris: 1400 | fitur: 9
Numerik : ['usia', 'masa_kerja_bulan', 'jam_lembur_bulanan', 'jumlah_training_setahun', 'proyek_selesai', 'skor_umpan_balik_360', 'hari_absen_setahun']
Kategori: ['departemen', 'pendidikan']

Distribusi target:
kategori_kinerja
Rendah    560
Sedang    700
Tinggi    140


,usia,masa_kerja_bulan,departemen,pendidikan,jam_lembur_bulanan,jumlah_training_setahun,proyek_selesai,skor_umpan_balik_360,hari_absen_setahun,kategori_kinerja
0,23,1,IT,Diploma,26.2,3,6,2.53,4,Rendah
1,49,74,HR,S1,17.0,2,5,3.10,9,Rendah
2,44,58,Operations,SMA,29.1,7,6,3.30,6,Sedang


## 2. EDA Singkat

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df[TARGET].value_counts().reindex(ORDER).plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Distribusi kategori kinerja"); axes[0].tick_params(axis="x", rotation=0)

sns.boxplot(data=df, x=TARGET, y="skor_umpan_balik_360", order=ORDER, ax=axes[1])
axes[1].set_title("Skor 360 vs kategori")

sns.scatterplot(data=df, x="proyek_selesai", y="skor_umpan_balik_360",
                hue=TARGET, hue_order=ORDER, alpha=0.5, ax=axes[2])
axes[2].set_title("Proyek vs skor 360")
plt.tight_layout(); plt.show()

## 3. Split & Pipeline

In [4]:
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_COLS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT_COLS),
])
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (1050, 9) | Test: (350, 9)


## 4. Perbandingan Model (CV)

Kelas tidak seimbang → skor utama **macro-F1** (tiap kelas setara).

In [5]:
candidates = {
    "Dummy (prior)": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=SEED),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    f1m = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    rows.append({"model": name, "f1_macro_cv": f1m.mean(), "f1_macro_std": f1m.std(),
                 "accuracy_cv": acc.mean()})

hasil_cv = pd.DataFrame(rows).sort_values("f1_macro_cv", ascending=False).reset_index(drop=True)
print(hasil_cv.round(4).to_string(index=False))

                 model  f1_macro_cv  f1_macro_std  accuracy_cv
   Logistic Regression       0.5575        0.0410       0.6425
Hist Gradient Boosting       0.5012        0.0527       0.5886
           Extra Trees       0.4950        0.0215       0.5851
         Random Forest       0.4693        0.0426       0.6038
         Dummy (prior)       0.2222        0.0000       0.5000


## 5. Tuning Model Teratas

In [6]:
top_name = hasil_cv[hasil_cv["model"] != "Dummy (prior)"].iloc[0]["model"]
print("Model teratas:", top_name)

GRIDS = {
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    "Random Forest": {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 6, 10],
                      "clf__min_samples_leaf": [1, 5]},
    "Extra Trees": {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 6, 10]},
    "Hist Gradient Boosting": {"clf__learning_rate": [0.03, 0.1], "clf__max_iter": [200, 400],
                               "clf__max_depth": [None, 3]},
}
base = Pipeline([("prep", preprocess), ("clf", candidates[top_name])])
grid = GRIDS.get(top_name, {})
if grid:
    search = GridSearchCV(base, grid, scoring="f1_macro",
                          cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED), n_jobs=-1)
    search.fit(X_train, y_train)
    best_pipe = search.best_estimator_
    print("Best params:", search.best_params_)
    print("Best CV macro-F1:", round(search.best_score_, 4))
else:
    best_pipe = base.fit(X_train, y_train)

Model teratas: Logistic Regression


Best params: {'clf__C': 1.0}
Best CV macro-F1: 0.5572


## 6. Evaluasi Holdout (termasuk metrik ordinal)

`MAE ordinal`: rata-rata |prediksi − aktual| pada skala 1–3.
Prediksi "Sedang" untuk karyawan "Tinggi" dihitung sebagai kesalahan 1,
bukan kesalahan penuh seperti pada akurasi.

In [7]:
pred_test = best_pipe.predict(X_test)
labels = list(best_pipe.classes_)

metrics = {
    "accuracy": accuracy_score(y_test, pred_test),
    "f1_macro": f1_score(y_test, pred_test, average="macro"),
    "f1_weighted": f1_score(y_test, pred_test, average="weighted"),
    "cohen_kappa": cohen_kappa_score(y_test, pred_test),
    "mae_ordinal": mean_absolute_error(
        y_test.map(LABEL2NUM), pd.Series(pred_test).map(LABEL2NUM)),
}
for k, v in metrics.items():
    print(f"{k:>12}: {v:.4f}")

print("\n" + classification_report(y_test, pred_test, zero_division=0))

cm = confusion_matrix(y_test, pred_test, labels=labels)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Prediksi"); plt.ylabel("Aktual"); plt.title("Confusion matrix (test)")
plt.tight_layout(); plt.show()

    accuracy: 0.6971
    f1_macro: 0.6357
 f1_weighted: 0.6910
 cohen_kappa: 0.4570
 mae_ordinal: 0.3086

              precision    recall  f1-score   support

      Rendah       0.74      0.66      0.70       140
      Sedang       0.67      0.79      0.73       175
      Tinggi       0.68      0.37      0.48        35

    accuracy                           0.70       350
   macro avg       0.70      0.61      0.64       350
weighted avg       0.70      0.70      0.69       350



## 7. Interpretasi — Permutation Importance

In [8]:
perm = permutation_importance(best_pipe, X_test, y_test, scoring="f1_macro",
                              n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.DataFrame({"fitur": X_test.columns,
                    "turunnya_f1_macro": perm.importances_mean}).sort_values(
    "turunnya_f1_macro", ascending=False).reset_index(drop=True)
print(imp.round(4).to_string(index=False))

plt.figure(figsize=(8, 5))
sns.barplot(data=imp.head(10), x="turunnya_f1_macro", y="fitur", color="#4C78A8")
plt.title("Permutation importance (macro-F1, test set)")
plt.tight_layout(); plt.show()

                  fitur  turunnya_f1_macro
   skor_umpan_balik_360             0.1790
             departemen             0.1596
             pendidikan             0.0708
jumlah_training_setahun             0.0356
         proyek_selesai             0.0255
     jam_lembur_bulanan             0.0199
       masa_kerja_bulan             0.0130
                   usia             0.0085
     hari_absen_setahun            -0.0047


## 8. Simpan Artifact

In [9]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipe, ARTIFACT_DIR / "performance_model.joblib")

metadata = {
    "versi_model": "v1",
    "model": top_name,
    "target": TARGET,
    "kelas": ORDER,
    "fitur_numerik": NUM_COLS,
    "fitur_kategorikal": CAT_COLS,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "dilatih_pada": pd.Timestamp.utcnow().isoformat(),
    "metrik_holdout_test": {k: round(float(v), 4) for k, v in metrics.items()},
    "dataset": str(DATA_PATH),
    "dataset_sintetis": True,
    "catatan": ("Target ordinal. Utamakan MAE ordinal & macro-F1. Jangan pakai "
                "untuk keputusan PHK."),
}
(ARTIFACT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print("Tersimpan:", ARTIFACT_DIR)
print(json.dumps(metadata["metrik_holdout_test"], indent=2))

Tersimpan: /workspace/project/hr-intelligence-system/models/performance/v1
{
  "accuracy": 0.6971,
  "f1_macro": 0.6357,
  "f1_weighted": 0.691,
  "cohen_kappa": 0.457,
  "mae_ordinal": 0.3086
}
